# PyLake Data management



Note:
* The examples use real files provided during the PyLake development work
* PyLake works with NumPy arrays and Xarray objects
* For Xarray inputs, time and depth dimensions should be named `time` and `depth`
* The workflow stays close to the original PyLake demo notebook


Import the necessary packages:

In [ ]:
from pathlib import Path

import pylake
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Example files

In [ ]:
downloads = Path.home() / "Downloads"

rsk_path = downloads / "236825_20260813_1313.rsk"
kor_path = downloads / "Kor Measurement File Export - 072325 141901.csv"
tob_path = downloads / "20210108__11081441_1.TOB"

print(rsk_path.exists(), rsk_path.name)
print(kor_path.exists(), kor_path.name)
print(tob_path.exists(), tob_path.name)


# RBR RSK example

The RSK file is an SQLite-based RBR measurement file.

`read_rsk` reads the measurement channels and returns them as an Xarray Dataset.


In [ ]:
rsk = pylake.read_rsk(rsk_path)
rsk


In [ ]:
list(rsk.data_vars)


In [ ]:
rsk["temp14"].plot(figsize=(10, 4))
plt.title("RBR temperature")
plt.show()


# KOR CSV example

`read_kor` reads the measurement table, builds the time coordinate and keeps the numeric measurement columns.


In [ ]:
kor = pylake.read_kor(kor_path)
kor


In [ ]:
Temp = kor["TEMP °C"].values
depth = kor["DEPTH M"].values

valid = np.isfinite(Temp) & np.isfinite(depth)

Temp = Temp[valid]
depth = depth[valid]


In [ ]:
plt.plot(Temp, depth)
plt.gca().invert_yaxis()
plt.xlabel("Temperature (°C)")
plt.ylabel("Depth (m)")
plt.show()


Let's start with methods that only need temperature and depth:

* Thermocline
* Seasonal thermocline
* Epilimnion and hypolimnion depth
* Mixed layer depth
* Buoyancy frequency


In [ ]:
thermoD, thermoInd = pylake.thermocline(Temp, depth)
epilimnion, hypolimnion = pylake.metalimnion(Temp, depth)
SthermoD, SthermoInd = pylake.seasonal_thermocline(Temp, depth)
hML = pylake.mixed_layer(Temp, depth, threshold=0.4)
n2 = pylake.buoyancy_freq(Temp, depth)

print("Thermocline:", thermoD)
print("Seasonal thermocline:", SthermoD)
print("Epilimnion:", epilimnion)
print("Hypolimnion:", hypolimnion)
print("Mixed layer:", hML)


# Sea & Sun TOB example

The TOB file contains instrument information followed by a CTD measurement table.

`read_tob` reads the measurements and returns them as an Xarray Dataset.


In [ ]:
tob = pylake.read_tob(tob_path)

tob


In [ ]:
plt.plot(tob.temperature, tob.pressure)
plt.gca().invert_yaxis()
plt.xlabel("Temperature (°C)")
plt.ylabel("Pressure (dbar)")
plt.show()


Pressure is kept as pressure here. It is not silently renamed to depth.


# DataLakes example

The DataLakes reader supports JSON, NetCDF and ZIP archives.


In [ ]:
datalakes_files = sorted(downloads.glob("*datalakesdownload*.zip"))

for path in datalakes_files:
    print(path.name)


In [ ]:
if datalakes_files:
    datalakes = pylake.read_datalakes(datalakes_files[0])
    datalakes


# Methods requiring additional information

Some PyLake methods also need information such as:

* Bathymetry
* Fetch length
* Friction velocity
* Layer density


In [ ]:
bthA = np.array([100, 90, 86, 82, 20, 1])
bthD = np.array([0, 2.3, 2.5, 4.2, 5.8, 7])
Lo = 50
ustar = 0.5


The same workflow as the original PyLake demo can then be used for methods such as:

* Wedderburn number
* Schmidt stability
* Heat content
* Seiche period
* Lake number

The scientific inputs should come from the dataset or known lake metadata rather than being invented from the measurement file.


# Summary

This notebook keeps the original PyLake tutorial style while showing how real acquisition formats enter the library:

* RBR `.rsk`
* KOR `.csv`
* Sea & Sun `.TOB`
* DataLakes JSON / NetCDF / ZIP
